# Seasonality in CTA bus ridership

What ridership does over the course of a year, once the trend is removed and holiday weeks are
held out. Split out of `exploration.ipynb`, which this notebook depends on.

**Run `exploration.ipynb` and then `holidays.ipynb` first** — they write the three files read
below.

### Outline
0. Setup: load the cleaned daily data, the route inventory and the holiday calendar
1. Building a seasonal index, and the profile by era
2. Do the eras describe the same season?
3. How much do holiday weeks matter?
4. Is one system-wide profile enough?

## 0. Setup

Imports and plotting parameters are the same block as in `exploration.ipynb`.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

# Okabe-Ito: the published colour-vision-deficiency-safe categorical set.
BLUE, ORANGE, GREEN, PURPLE = '#0072B2', '#D55E00', '#009E73', '#CC79A7'
GRAY, INK = '#C9C9C9', '#333333'

# Ridership regimes. These are a COLOUR SCHEME for reading charts over time, not an
# analysis grouping: any analysis that needs a particular window defines it locally and
# prints what it used. 2020 gets its own colour because it is not comparable to anything
# else; the recovery years are lighter shades of it because the system has not returned
# to the pre-2020 level; 2025-present is distinct because the Frequent Network rollout
# begins 2025-03-23, so those years are not a clean baseline for anything.
ERAS = [('pre-2020',     2001, 2019, BLUE),
        ('2020',         2020, 2020, ORANGE),
        ('2021-2022',    2021, 2022, '#EE8A4E'),
        ('2023-2024',    2023, 2024, '#F5BE99'),
        ('2025-present', 2025, 2026, GREEN)]
ERA_ORDER = [e[0] for e in ERAS]
ERA_COLOR = {e[0]: e[3] for e in ERAS}

def era(year):
    """Map a calendar year to its ridership regime."""
    for name, lo, hi, _ in ERAS:
        if lo <= year <= hi:
            return name
    return None

# The 20 Frequent Network routes, as CTA labels them. Defined here because several
# sections need it, including the corridor check in section 2.
FREQ = ['J14', '4', '9', '12', '20', '34', '47', '49', '53', '54',
        '55', '60', '63', '66', '72', '77', '79', '81', '82', '95']

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#9A9A9A', 'axes.grid': True,
    'grid.color': '#E8E8E8', 'grid.linewidth': 0.8,
})
fmt_riders = FuncFormatter(lambda v, _: f'{v*1e-6:.1f}M' if v >= 1e6 else f'{v*1e-3:.0f}k')

In [ ]:
from pathlib import Path

DERIVED = Path('data/derived')
for f in ('daily.csv', 'route_inventory.csv', 'holiday_calendar.csv'):
    if not (DERIVED / f).exists():
        raise FileNotFoundError(
            f'{DERIVED / f} is missing. Run exploration.ipynb (daily.csv, '
            'route_inventory.csv) and then holidays.ipynb (holiday_calendar.csv) first.')

d = pd.read_csv(DERIVED / 'daily.csv',
                dtype={'route': str, 'corridor': str},
                parse_dates=['date', 'week'])
inv = pd.read_csv(DERIVED / 'route_inventory.csv', dtype={'route': str}, index_col='route')
cal = pd.read_csv(DERIVED / 'holiday_calendar.csv', parse_dates=['date'])

print(f'daily rows        : {len(d):,}   routes {d.route.nunique()}')
print(f'route inventory   : {len(inv):,} routes')
print(f'holiday calendar  : {len(cal):,} dates, {int(cal.holiday.sum()):,} flagged as holidays')

# Weekly system totals, same construction as exploration.ipynb section 1.c.
wk = (d.groupby('week')
        .agg(rides=('rides', 'sum'), days=('date', 'nunique'), routes=('route', 'nunique'))
        .reset_index())
wk['partial'] = wk.days < 7
print(f'weeks             : {len(wk):,}')

# Which two year-groupings this notebook compares in section 2. Named here, not inherited:
# the era bands in exploration.ipynb are a chart colour scheme, and which years can legitimately
# be pooled into one seasonal profile is the open question this notebook exists to answer.
# 2025-present is deliberately not the comparison era -- the Frequent Network rollout starts
# 2025-03-23, and 2026 contributes no weeks at all (no full centred window).
COMPARE = ('pre-2020', '2023-2024')

## 1. Building a seasonal index

A before/after comparison needs to know what ridership does over a year anyway. Three things
make that harder than reading a monthly average, and this notebook deals with them one at a time.

**Trend contaminates a same-week ratio.** Dividing week *w* of one year by week *w* of the
previous year gives the seasonal difference *plus* that year's growth. So the trend is removed
first: each week is divided by a **centred 53-week moving average**, which is a full year wide
and symmetric around the week it describes. What is left is a multiplicative *seasonal index* —
1.0 means "typical for this year", 0.9 means "10% below the year's own level".

**Holiday weeks are not seasonal in the week-number sense.** Christmas moves between ISO weeks
51, 52 and 1 depending on the year, so a week-number average silently mixes holiday and normal
weeks. The 152 holiday dates found in `holidays.ipynb` are used to hold those weeks out of the
profile; the holiday effect is then applied separately, where it can be measured properly.

**Which years to include is an empirical question**, not a choice to make up front. The profile
is computed per era and the eras are compared. If they agree, pool them; if they don't, only the
recent ones are admissible and the resulting loss of precision is a real constraint worth knowing
about before the comparison is designed.

In [ ]:
sw = wk[['week', 'rides', 'days', 'routes']].copy()
sw['year'] = sw.week.dt.year
sw['era']  = sw.year.map(era)
sw['woy']  = sw.week.dt.isocalendar().week.astype(int)     # ISO week number, 1..53

# A week is a holiday week if any of the 152 dates from holidays.ipynb falls inside it.
hol_mondays = (cal.loc[cal.holiday, 'date']
                  - pd.to_timedelta(cal.loc[cal.holiday, 'date'].dt.weekday, unit='D'))
sw['holiday_week'] = sw.week.isin(set(hol_mondays))

# Detrend. min_periods=53 means no partial windows, so 26 weeks at each end have no index.
sw['trend'] = sw.rides.rolling(53, center=True, min_periods=53).mean()
sw['index'] = sw.rides / sw.trend

print(f'weeks                        : {len(sw):,}')
print(f'  with a centred trend value : {int(sw["index"].notna().sum()):,} '
      f'({int(sw["index"].isna().sum())} at the two ends have no full window)')
print(f'  holiday weeks              : {int(sw.holiday_week.sum()):,}')
print(f'  usable for the profile     : {int((sw["index"].notna() & ~sw.holiday_week).sum()):,}')

woy_n = sw.woy.value_counts().sort_index()
print(f'\nISO week numbers present: {woy_n.index.min()}..{woy_n.index.max()}   '
      f'week 53 occurs {int(woy_n.get(53, 0))} times (ISO years with 53 weeks)')

In [ ]:
def profile(frame):
    """Median seasonal index by ISO week number, with quartiles and a count."""
    g = frame.groupby('woy')['index']
    return pd.DataFrame({'median': g.median(), 'q25': g.quantile(.25),
                         'q75': g.quantile(.75), 'n': g.size()})

usable = sw['index'].notna() & ~sw.holiday_week
profiles = {name: profile(sw[usable & (sw.era == name)]) for name in ERA_ORDER}

fig, ax = plt.subplots(figsize=(11, 4.6))
for name in ERA_ORDER:
    p = profiles[name]
    if p.empty:
        continue
    ax.plot(p.index, p['median'], lw=1.6, color=ERA_COLOR[name], label=f'{name}')
for name in COMPARE:
    p = profiles[name]
    ax.fill_between(p.index, p.q25, p.q75, color=ERA_COLOR[name], alpha=0.18, linewidth=0)

ax.axhline(1, color=INK, lw=0.8, ls=':')
ax.set_xlabel('ISO week of year'); ax.set_ylabel('seasonal index (1.0 = typical for that year)')
ax.set_title('Seasonal profile by era — holiday weeks held out, bands are the IQR',
             loc='left', fontsize=11)
ax.legend(frameon=False, ncol=4, loc='lower center')
plt.show()

In [ ]:
# Holding holiday weeks out leaves gaps: some ISO weeks contain a holiday in every year.
cover = pd.DataFrame({'all_weeks': profile(sw[sw['index'].notna()])['n'],
                      'non_holiday': profile(sw[usable])['n']}).fillna(0).astype(int)
missing = cover.index[cover.non_holiday == 0]

print(f'ISO week numbers with no non-holiday observation at all: {list(missing)}')
print('These cannot be estimated from the held-out profile and need the holiday')
print('adjustment from holidays.ipynb instead.\n')
print('thinnest coverage among the weeks that do survive:')
print(cover[cover.non_holiday > 0].nsmallest(8, 'non_holiday').to_string())

## 2. Do the groupings describe the same season?

The two groupings named in `COMPARE` above, compared week by week. If they disagree, a single
seasonal profile cannot be pooled across them.

In [ ]:
# Do the two groupings describe the same season? Compare them week by week.
A, B = COMPARE
a, b = profiles[A], profiles[B]
common = a.index.intersection(b.index)
diff = (a.loc[common, 'median'] - b.loc[common, 'median'])

print(f'weeks compared          : {len(common)}')
print(f'correlation             : {np.corrcoef(a.loc[common, "median"], b.loc[common, "median"])[0, 1]:.3f}')
print(f'mean |difference|       : {diff.abs().mean():.4f}')
print(f'largest |difference|    : {diff.abs().max():.4f} at week {diff.abs().idxmax()}')
print(f'seasonal range, {A:<14}: {a["median"].min():.3f} .. {a["median"].max():.3f}')
print(f'seasonal range, {B:<14}: {b["median"].min():.3f} .. {b["median"].max():.3f}')
print(f'\nweeks per era used:')
for name in ERA_ORDER:
    p = profiles[name]
    print(f'  {name:<14} {int(p["n"].sum()):>4} weeks across {p.index.size} week-numbers')

print('\nthe ten weeks where the two groupings disagree most:')
print(pd.DataFrame({A: a.loc[common, 'median'], B: b.loc[common, 'median'],
                    'diff': diff}).reindex(diff.abs().sort_values(ascending=False).index)
        .head(10).round(3).to_string())

## 3. How much do holiday weeks matter?

The same profile computed with holiday weeks left in, against the one with them held out. The
gap is the reason they are held out rather than averaged over.

In [ ]:
with_hol = profile(sw[sw['index'].notna()])
without   = profile(sw[usable])
gap = (with_hol['median'] - without['median']).dropna()

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(without.index, without['median'], lw=1.6, color=BLUE, label='holiday weeks held out')
ax.plot(with_hol.index, with_hol['median'], lw=1.3, color=ORANGE, ls='--',
        label='holiday weeks included')
ax.axhline(1, color=INK, lw=0.8, ls=':')
ax.set_xlabel('ISO week of year'); ax.set_ylabel('seasonal index')
ax.set_title('Effect of leaving holiday weeks in the seasonal profile (all years)',
             loc='left', fontsize=11)
ax.legend(frameon=False)
plt.show()

print('weeks most affected by including holiday weeks:')
print(gap.reindex(gap.abs().sort_values(ascending=False).index).head(8).round(3).to_string())

## 4. Is one system-wide profile enough?

If routes have different seasonal shapes, a single system profile will mis-adjust individual
corridors. Each route's own profile is correlated against the system's.

In [ ]:
rwk = d.pivot_table(index='week', columns='route', values='rides', aggfunc='sum')
ridx = rwk / rwk.rolling(53, center=True, min_periods=53).mean()

ok_weeks = sw.set_index('week').loc[ridx.index, 'holiday_week'].to_numpy()
rlong = (ridx[~ok_weeks].stack().rename('index').reset_index()
           .assign(woy=lambda t: t.week.dt.isocalendar().week.astype(int)))

sys_prof = without['median']
corrs, ns = {}, {}
for r, grp in rlong.groupby('route'):
    p = grp.groupby('woy')['index'].median()
    common = p.index.intersection(sys_prof.index)
    ns[r] = len(grp)
    corrs[r] = np.corrcoef(p[common], sys_prof[common])[0, 1] if len(common) > 20 else np.nan

cs = pd.Series(corrs).rename('corr_with_system')
print(f'routes with a profile            : {int(cs.notna().sum())} of {len(cs)}')
print(f'  too few weeks to compare       : {int(cs.isna().sum())}')
print(f'median correlation with system   : {cs.median():.3f}')
print(f'  quartiles                      : {cs.quantile(.25):.3f} .. {cs.quantile(.75):.3f}')
print(f'  routes below 0.5               : {int((cs < 0.5).sum())}')

print('\nleast like the system:')
print(pd.DataFrame({'corr': cs, 'name': inv.name, 'riders/day': inv['mean'].round(0)})
        .dropna(subset=['corr']).nsmallest(10, 'corr').to_string())
print('\nFrequent Network routes:')
print(pd.DataFrame({'corr': cs, 'name': inv.name})
        .reindex(FREQ).dropna(subset=['corr']).sort_values('corr').round(3).to_string())

## Where this leaves us

### Established

- Seasonal range: 0.944–1.113 pre-2020; 0.835–1.125 across 2023-2024.
- pre-2020 and 2023-2024 correlate 0.75 over the 47 week-numbers they share. Largest gaps at
  week 3 (0.944 vs 0.835) and week 35 (1.012 vs 1.112) (§2).
- Holiday weeks have to be held out, not averaged over (§3).
- 32 of 136 routes correlate below 0.5 with the system profile; the 20 Frequent Network routes
  run 0.57–0.97 (§4).

### Decisions still open

1. Which years feed the profile. Usable weeks: 853 pre-2020, 92 in 2023-2024, 43 in
   2025-present, 0 from 2026 (no full centred window).
2. `COMPARE` in §0 is currently pre-2020 vs 2023-2024.
3. Routes with inverted seasonality sit in the control group untreated.